In [1]:
from glob import glob
from collections import Counter
from datasets import load_dataset, Dataset, concatenate_datasets
from glob import glob
from tqdm import tqdm

import json
import os

/opt/conda/envs/sppo/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
cache_dir = "../results_synthetic/from-preference__SlimOrca-Dedup/cache"

lst_cache_files = glob(f"{cache_dir}/*.json")
lst_cache_files = sorted(lst_cache_files, key=lambda x: int(x.split("/")[-1].split(".")[0]))
results = []
for cache_file in tqdm(lst_cache_files):
    with open(cache_file, "r") as f:
        results.append(json.load(f))

100%|██████████| 363491/363491 [01:53<00:00, 3199.33it/s]


In [4]:
results[0]

[{'content': 'You are an AI assistant. You will be given a task. You must generate a detailed and long answer.',
  'role': 'system'},
 {'content': 'Write an article based on this "A man has been charged with murder and attempted murder after a woman and the man she was on a date with were stabbed at a restaurant in Sydney, Australia."',
  'role': 'user'},
 {'role': 'assistant',
  'content': ' **Title: Sydney Restaurant Stabbing: Man Charged with Murder and Attempted Murder**\n\nIn a shocking turn of events, a man has been charged with murder and attempted murder following a brutal knife attack at a bustling restaurant in Sydney, Australia. The incident, which has left the city in a state of disbelief, occurred during the evening hours when patrons were enjoying their meals and the company of their friends and loved ones.\n\n**The Incident**\n\nThe Sydney Police Department responded to an emergency call from a popular restaurant in the heart of the city. Upon arrival, officers discovere

In [8]:
def fmt_openAI_to_shareGPT(messages):
    conversations = []
    for msg in messages:
        if msg["role"] == "user":
            from_ = "human"
        elif msg["role"] == "assistant":
            from_ = "gpt"
        elif msg["role"] == "system":
            from_ = "system"
        else:
            raise ValueError(f"Invalid message role: {msg['role']}")
            
        conversations.append({"from": from_, "value": msg["content"]})
        
    return conversations

ds = Dataset.from_dict(
    {
        "messages": results,
        "conversations": [fmt_openAI_to_shareGPT(messages) for messages in results],
        "model": ["mistralai/Mistral-Large-Instruct-2407"] * len(results),
    }
)

In [10]:
ds.push_to_hub("slm-research-vn/SlimOrca-Dedup", private=True)

Uploading the dataset shards: 100%|██████████| 3/3 [00:49<00:00, 16.34s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/slm-research-vn/SlimOrca-Dedup/commit/7f199610be5f3ab3081a2e8da4ce4c462d864691', commit_message='Upload dataset', commit_description='', oid='7f199610be5f3ab3081a2e8da4ce4c462d864691', pr_url=None, pr_revision=None, pr_num=None)